In [26]:
import pandas as pd

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [27]:
# T1-3
pattern = r'\bpâni\b'
wordCount = train['text'].str.count(pattern).sum() + test['text'].str.count(pattern).sum()
task1ans = wordCount

punctuations = r"[,.\-?!'\"“”„”]"
graiMold = train[train['label'] == 'graiul moldovenesc']['text'] # Filters
graiBana = train[train['label'] == 'graiul bănățean']['text']
graiMold['count'] = graiMold.str.count(punctuations) # Counting
graiBana['count'] = graiBana.str.count(punctuations)
Difference = graiMold['count'].mean() - graiBana['count'].mean() # Difference
task2ans = round(abs(Difference) , 2)

diacritics = r"[ăâîșțĂÂÎȘȚ]"
test['T3'] = test['text'].str.count(diacritics)

In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

le = LabelEncoder()

y_train = le.fit_transform(train['label'])
X_train = train['text']
X_test = test['text']

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words=None, analyzer='char')),
    ('model', LogisticRegression(random_state=42, solver='saga', max_iter=2000))
])

parameters = {
    'tfidf__max_features': randint(1000, 3000),
    'tfidf__ngram_range': [(2, 4), (2, 5), (3, 5)], 
    'model__C': uniform(0.1, 10.0),
    'model__l1_ratio': uniform(0, 1)
}

rs = RandomizedSearchCV(
pipeline,
    param_distributions=parameters,
    n_iter=40,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1,
    random_state=42
)
rs.fit(X_train, y_train)
model = rs.best_estimator_
predictions = model.predict(X_test)
predictions = le.inverse_transform(predictions)

Fitting 5 folds for each of 40 candidates, totalling 200 fits


In [31]:
rows = []
rows.append({
    'subtaskID': 1,
    'datapointID': 1,
    'answer': task1ans
})
rows.append({
    'subtaskID': 2,
    'datapointID': 1,
    'answer': task2ans
})
for id, t3 in zip(test['ID'], test['T3']):
    rows.append({
    'subtaskID': 3,
    'datapointID': id,
    'answer': t3
})
for id, pred in zip(test['ID'], predictions):
    rows.append({
    'subtaskID': 4,
    'datapointID': id,
    'answer': pred
})
sub = pd.DataFrame(rows)
sub.to_csv('submission.csv', index=False)